# AI Writing Coach

This notebook is your reference for everything we built in class.

---

### What We Built
A Streamlit web app that:
- Takes text input from a user
- Rewrites it in different styles (Academic, Professional, Casual, LinkedIn)
- Analyzes it and returns scores as structured JSON
- Handles errors without crashing

### The Core Concept
We never changed the model. We only changed the prompts.
That is Prompt Engineering.

---
## Part 1  Install Libraries

We need two libraries:
- `streamlit`  builds the web UI
- `groq`  connects to the Groq API to call the LLM

We use Groq because it's free and fast. The model we use is `llama-3.3-70b-versatile`.

In [ ]:
pip install streamlit groq

---
## Part 2  Imports

- `streamlit as st` , everything UI-related goes through `st`
- `Groq` , the client that talks to the API
- `json` , we need this to parse structured output (explained later)

In [ ]:
import streamlit as st
from groq import Groq
import json

---
## Part 3 , Personas (System Prompts)

This is the most important part of the whole app.

A **persona** is just a system prompt , instructions you give the model
before the user's message arrives. The model reads it first and uses it
to shape every response.

We have four personas. The model is identical for all four.
Only the system prompt changes. That's what produces different output.

**Why are the prompts 4-5 sentences long?**
A 6-word instruction gives the model direction but no texture.
Longer prompts give it tone, constraints, and style , so the output
actually sounds different, not just labeled differently.

In [ ]:
PERSONAS = {
    "Academic": """You are an academic writing assistant. 
Rewrite the given text in a formal academic tone suitable for research papers or university assignments. 
Use precise vocabulary, passive voice where appropriate, and structured sentences. 
Avoid contractions, casual language, and first-person unless necessary. 
Cite reasoning clearly and maintain an objective, analytical stance.""",

    "Professional": """You are a professional business communication expert. 
Rewrite the given text for workplace settings - emails, reports, or internal communication. 
Be clear, concise, and respectful. Use active voice. 
Remove filler words and emotional language. 
The output should sound like it came from a senior professional who respects the reader's time.""",

    "Casual": """You are a friendly writing assistant. 
Rewrite the given text in a natural, relaxed tone - like texting a friend or writing an informal message. 
Use simple words, contractions, and a warm conversational flow. 
Keep it genuine, not performative. Avoid sounding like a corporate email.""",

    "LinkedIn": """You are a LinkedIn content strategist who writes posts that get real engagement. 
Rewrite the given text as a LinkedIn post with a strong hook that stops the scroll. 
Use short punchy paragraphs, one idea per line. 
Make it personal and specific - avoid generic motivational language. 
End with a question or observation that invites comments.
Add proper multiple emojis as a supporting character."""
}

---
## Part 4 — Understanding the API Call

This is the core of how you talk to an LLM.

```python
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system",    "content": PERSONAS[persona]},
        {"role": "user",      "content": user_text}
    ]
)
```

**Line by line:**

| Code | What it does |
|---|---|
| `client.chat.completions.create(...)` | Sends a request to Groq. 'Completion' means: give me what comes next after this input. |
| `model=...` | Which AI brain to use. Swap this string to use a different model — nothing else changes. |
| `messages=[...]` | The full conversation history sent every call. LLMs have no memory — you send everything each time. |
| `role: system` | Master instruction. The model reads this first. It sets behavior for the entire response. |
| `role: user` | What the human typed. The actual input. |
| `response.choices[0].message.content` | The model's reply. `choices[0]` = first result. `.message.content` = the actual text. |

**The three roles you'll encounter:**

| Role | Purpose |
|---|---|
| `system` | Tells the model who it is and how to behave |
| `user` | What the human sent |
| `assistant` | What the model previously replied — used in multi-turn chat to give the model memory |

In this app we only use `system` and `user` because each request is independent.

---
## Part 5 — Test a Basic API Call

Before building the full app, verify your connection works.

Get your free API key at: https://console.groq.com/keys

**Important:** Never hardcode your API key in code you share or push to GitHub.
In our app we take it as a password input from the user instead.

In [ ]:
# Paste your key here ONLY for local testing , never commit this to GitHub
GROQ_API_KEY = "paste_your_key_here"

client = Groq(api_key=GROQ_API_KEY)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)

print(response.choices[0].message.content)

---
## Part 6 — Test Personas

Same input. Same model. Different system prompt. Different output.
Run this cell and see what changes.

In [ ]:
test_text = "i done my project last night it was hard but i think its okay the results were not that good but i tried my best"

for persona_name, persona_prompt in PERSONAS.items():
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": persona_prompt},
            {"role": "user",   "content": test_text}
        ]
    )
    
    print(f"\n{'='*50}")
    print(f"PERSONA: {persona_name}")
    print('='*50)
    print(response.choices[0].message.content)

---
## Part 7 — Structured Output (JSON Mode)

LLMs normally return prose — text a human reads.
But applications need **data**, not text.

Structured output means you instruct the model to return JSON
instead of a paragraph. Your code then parses that JSON and uses
the values — like displaying them as metric cards in the UI.

**How to force JSON output:**
1. Tell the model explicitly in the prompt: `Return ONLY valid JSON. No markdown, no explanation.`
2. Give it the exact schema you want filled in
3. Parse the response with `json.loads()`

This is how real products are built on top of LLMs — the model becomes
a data extraction or scoring engine, not just a chatbot.

In [ ]:
sample_text = "Hey so basically I was thinking we could maybe do the meeting on thursday if thats cool. let me know. also bring the report if u have it done"

analysis_prompt = f"""
Analyze the following text.

Return ONLY valid JSON. No markdown, no explanation, nothing else.

{{
    "clarity": 0,
    "formality": 0,
    "readability": 0
}}

Scores are integers from 0 to 10.

Text:
{sample_text}
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": analysis_prompt}
    ]
)

raw = response.choices[0].message.content
print("Raw model output:")
print(raw)

# Parse the JSON string into a Python dict
parsed = json.loads(raw)
print("\nParsed as Python dict:")
print(parsed)
print(f"\nClarity score: {parsed['clarity']}")

---
## Part 8 — Error Handling

Two things can go wrong and you need to handle them separately:

**1. API error** — network issue, wrong key, rate limit hit.
The call itself fails before you get any response.

**2. JSON parse error** — the API call succeeded but the model
returned text instead of valid JSON. `json.loads()` fails.

Catching them separately lets you show the user a specific,
useful message instead of a raw Python traceback.

**Rule:** Applications should never show users a raw error traceback.
Catch it, explain it simply, give them a way forward.

In [ ]:
def analyze_text(text):
    
    prompt = f"""
    Analyze the following text.
    Return ONLY valid JSON. No markdown, no explanation.
    {{"clarity": 0, "formality": 0, "readability": 0}}
    Scores are integers from 0 to 10.
    Text: {text}
    """
    
    try:
        # This can fail if the API is down or the key is wrong
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )
        
        result = response.choices[0].message.content
        
        # This can fail if the model returned prose instead of JSON
        parsed = json.loads(result)
        return parsed
    
    except json.JSONDecodeError:
        print("Model returned something that isn't valid JSON. Try again.")
        return None
    
    except Exception as e:
        print(f"API error: {e}")
        return None


result = analyze_text("i tried to finish my assignment but it was confusing and i gave up")
print(result)

---
## Part 9 — The Full App

Everything above combined into the final `app.py`.

Save the code below as `app.py` and run it with:
```
streamlit run app.py
```

**Key decisions in the final app:**

| Decision | Why |
|---|---|
| API key as sidebar input | Never hardcode secrets. Each student uses their own key. |
| `get_client()` helper function | Client is created once and reused. Avoids repeating the same code in every button block. |
| Empty text check before API call | Don't waste an API call on empty input. Show a clear message instead. |
| `json.JSONDecodeError` caught separately | Different problem, different message. Specific errors = better UX. |
| `st.spinner(...)` on both buttons | Gives visual feedback while the API call runs — usually 1-3 seconds. |

In [ ]:
# Save this cell as app.py and run: streamlit run app.py

app_code = '''
import streamlit as st
from groq import Groq
import json

st.set_page_config(page_title="AI Writing Coach", layout="wide")
st.title("AI Writing Coach")

PERSONAS = {
    "Academic": """You are an academic writing assistant. 
Rewrite the given text in a formal academic tone suitable for research papers or university assignments. 
Use precise vocabulary, passive voice where appropriate, and structured sentences. 
Avoid contractions, casual language, and first-person unless necessary. 
Cite reasoning clearly and maintain an objective, analytical stance.""",

    "Professional": """You are a professional business communication expert. 
Rewrite the given text for workplace settings - emails, reports, or internal communication. 
Be clear, concise, and respectful. Use active voice. 
Remove filler words and emotional language. 
The output should sound like it came from a senior professional who respects the readers time.""",

    "Casual": """You are a friendly writing assistant. 
Rewrite the given text in a natural, relaxed tone - like texting a friend or writing an informal message. 
Use simple words, contractions, and a warm conversational flow. 
Keep it genuine, not performative. Avoid sounding like a corporate email.""",

    "LinkedIn": """You are a LinkedIn content strategist who writes posts that get real engagement. 
Rewrite the given text as a LinkedIn post with a strong hook that stops the scroll. 
Use short punchy paragraphs, one idea per line. 
Make it personal and specific - avoid generic motivational language. 
End with a question or observation that invites comments.
Add proper multiple emojis as a supporting character."""
}

st.sidebar.header("Settings")
api_key = st.sidebar.text_input("Groq API Key", type="password", placeholder="Paste your Groq API key here")
persona = st.sidebar.selectbox("Choose Persona", list(PERSONAS.keys()))

user_text = st.text_area("Enter your text", height=200, placeholder="Paste your text here...")

def get_client():
    if not api_key:
        st.warning("Please enter your Groq API key in the sidebar.")
        return None
    return Groq(api_key=api_key)

if st.button("Improve Writing", use_container_width=True):
    if not user_text.strip():
        st.warning("Please enter some text first.")
    else:
        client = get_client()
        if client:
            try:
                with st.spinner("Improving writing..."):
                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=[
                            {"role": "system", "content": PERSONAS[persona]},
                            {"role": "user",   "content": user_text}
                        ]
                    )
                st.subheader("Improved Version")
                st.write(response.choices[0].message.content)
            except Exception as e:
                st.error(f"API error: {e}")

st.divider()
st.subheader("Writing Analysis")

if st.button("Analyze Writing", use_container_width=True):
    if not user_text.strip():
        st.warning("Please enter some text first.")
    else:
        client = get_client()
        if client:
            analysis_prompt = f"""
Analyze the following text.
Return ONLY valid JSON. No markdown, no explanation, nothing else.
{{"clarity": 0, "formality": 0, "readability": 0}}
Scores are integers from 0 to 10.
Text: {user_text}
"""
            try:
                with st.spinner("Analyzing writing..."):
                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=[{"role": "user", "content": analysis_prompt}]
                    )
                result = response.choices[0].message.content
                st.code(result, language="json")
                parsed = json.loads(result)
                col1, col2, col3 = st.columns(3)
                col1.metric("Clarity",     parsed["clarity"])
                col2.metric("Formality",   parsed["formality"])
                col3.metric("Readability", parsed["readability"])
            except json.JSONDecodeError:
                st.warning("The model returned something that isnt valid JSON. Try again.")
            except Exception as e:
                st.error(f"API error: {e}")
'''

with open("app.py", "w") as f:
    f.write(app_code.strip())

print("app.py saved. Run it with: streamlit run app.py")

---
## Part 10 — Challenge (Try After Class)

Add these four new personas to the PERSONAS dictionary.
You don't touch anything else in the app — just add to the dict.

That's the point. Same app. Same model. Only the prompts change.

```python
"Resume Reviewer":    "You are a senior recruiter with 10 years experience...",
"Interview Coach":    "You are an interview coach who prepares candidates...",
"Email Writer":       "You are an email communication specialist...",
"Grammar Checker":    "You are a precise grammar and style editor..."
```

Write the full persona prompts yourself — 4 to 5 sentences each.
The more specific your instructions, the better the output.

---

## Quick Reference

| Concept | Where it appears in app |
|---|---|
| System prompt | `role: system` with PERSONAS dict |
| Zero-shot prompting | Writing analysis — no examples given, model figures it out |
| Structured output | JSON analysis returning clarity/formality/readability |
| Error handling | `try/except` blocks around every API call |
| API key security | `st.sidebar.text_input(..., type="password")` |

**Run the app:**
```bash
streamlit run app.py
```

**Groq API keys:** https://console.groq.com/keys  
**Streamlit docs:** https://docs.streamlit.io